# 05 — Embeddings de token vs. embeddings de oración


## De qué se trata este notebook

En el notebook 04 pedimos un vector por oración y no preguntamos de dónde salía. Aquí abrimos esa
caja. Vamos a comparar dos formas de convertir texto en vectores:

1. **Embeddings de token, con BERT** (`bert-base-uncased`, vía `transformers`). BERT produce **un
   vector por cada token** (aproximadamente, cada palabra o fragmento de palabra) y ese vector
   depende del **contexto**: a diferencia de un embedding estático tipo word2vec, la misma palabra
   tiene vectores distintos según la oración en la que aparezca. Para obtener un vector de la
   oración completa hay que combinar los de sus tokens de alguna forma —típicamente promediándolos.

2. **Embeddings de oración, con un dual encoder tipo SBERT** (`all-MiniLM-L6-v2`, vía
   `sentence-transformers`). Un solo vector por oración, producido por un modelo **entrenado
   específicamente** para que la distancia entre dos oraciones refleje qué tan parecidas son en
   significado.

La pregunta que vamos a responder con datos, no con intuición, es: **¿cuánto mejor es la segunda
opción, y por qué?** Para eso los evaluamos contra juicios de similitud hechos por humanos.

## Al terminar vas a poder

1. Explicar qué es la tokenización en subpalabras y por qué importa para el español.
2. Demostrar que los embeddings de BERT son contextuales, midiéndolo.
3. Implementar el *mean pooling* con máscara de atención y explicar para qué sirve la máscara.
4. Evaluar dos modelos de embeddings contra un estándar humano (STS-Benchmark) y diagnosticar
   por qué uno gana.

No necesitas cuenta ni llave de API: ambos modelos son abiertos y se descargan la primera vez que
los usas (`bert-base-uncased` pesa ~420 MB, `all-MiniLM-L6-v2` ~87 MB).

## 0. Preparación del entorno

> **Nota técnica (la misma del notebook 04).** `os.environ["USE_TF"] = "0"` debe ejecutarse
> **antes** de importar `transformers` o `sentence_transformers`, o el kernel puede caerse sin
> mensaje de error en equipos donde TensorFlow no está bien instalado.

In [ ]:
# --- Dependencias -------------------------------------------------------------
# Esta celda instala SOLO lo que falte, así que puedes correrla siempre.
# Es necesaria en Google Colab: ahí viene preinstalado `transformers`,
# pero NO `sentence-transformers`.
import importlib.util, subprocess, sys

EN_COLAB = "google.colab" in sys.modules

def asegurar(paquete, modulo=None):
    """Instala `paquete` solo si su módulo no está disponible.

    Usamos find_spec en vez de un `import` dentro de un try: find_spec NO ejecuta
    el módulo, y ejecutar `sentence_transformers` antes de fijar USE_TF puede
    tumbar el kernel (ver la nota técnica de la celda siguiente).
    """
    modulo = modulo or paquete.replace("-", "_")
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando {paquete} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

import os

for paquete, modulo in [("sentence-transformers", "sentence_transformers"),
                        ("transformers",          "transformers"),
                        ("scikit-learn",          "sklearn"),
                        ("seaborn",               "seaborn"),
                        ("matplotlib",            "matplotlib"),
                        ("pandas",                "pandas"),
                        ("scipy",                 "scipy")]:
    asegurar(paquete, modulo)

# `datasets` solo hace falta si NO tenemos al lado la copia local de los datos
# (ese es el caso en Colab, si abriste el notebook suelto sin el resto de la carpeta).
if not os.path.exists("stsbenchmark_muestra.csv"):
    asegurar("datasets")

print("Dependencias listas." + ("   (Google Colab detectado)" if EN_COLAB else ""))

In [ ]:
import os
os.environ["USE_TF"] = "0"          # SIEMPRE antes de importar transformers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import torch

from transformers import BertModel, BertTokenizer

import warnings
warnings.filterwarnings("ignore")

SEMILLA = 42
np.random.seed(SEMILLA)
print("Listo.")

In [ ]:
nombre_modelo = "bert-base-uncased"
tokenizador = BertTokenizer.from_pretrained(nombre_modelo)
bert = BertModel.from_pretrained(nombre_modelo)
bert.eval()          # modo evaluación: desactiva dropout, no vamos a entrenar

print(f"Vocabulario del tokenizador : {tokenizador.vocab_size:,} tokens")
print(f"Dimensión de cada embedding : {bert.config.hidden_size}")
print(f"Capas del modelo            : {bert.config.num_hidden_layers}")

## 1. Antes del vector: ¿qué ve realmente el modelo?

BERT no trabaja con palabras, sino con **tokens de subpalabra** (*WordPiece*). Su vocabulario es
finito (30,522 entradas), así que cualquier palabra que no esté en él se parte en fragmentos que
sí lo estén. Los fragmentos que continúan una palabra se marcan con `##`.

Además, BERT agrega dos tokens especiales a cada entrada: `[CLS]` al inicio y `[SEP]` al final.

In [ ]:
ejemplos = [
    "I deposited money at the bank.",
    "Embeddings are unbelievably useful for retrieval.",
    "La regresión logística ordinal",           # español: el modelo es solo de inglés
]

for texto in ejemplos:
    tokens = tokenizador.tokenize(texto)
    print(f"{texto}")
    print(f"   -> {tokens}")
    print(f"   -> {len(texto.split())} palabras se convirtieron en {len(tokens)} tokens\n")

In [ ]:
# Los tokens especiales que agrega el modelo:
codificado = tokenizador("I deposited money at the bank.", return_tensors="pt")
print(tokenizador.convert_ids_to_tokens(codificado["input_ids"][0]))

**Aquí está la explicación mecánica de algo que ya observamos.** En el notebook 04 vimos que
`all-MiniLM-L6-v2` daba resultados sin sentido en español. La tercera línea de la salida anterior
muestra por qué: `"regresión"` se parte en `reg / ##res / ##ion`, `"logística"` en
`log / ##istic / ##a`. El modelo no está representando conceptos en español; está armando
fragmentos ortográficos que en inglés significan otra cosa. Un modelo multilingüe tiene un
vocabulario que sí incluye las palabras en español, y por eso no las destroza.

**Consecuencia práctica adicional:** el costo de procesar un texto se cobra por *token*, no por
palabra. Un corpus en español procesado con un modelo en inglés cuesta más y rinde peor.

## 2. Los embeddings de BERT son **contextuales**: demostrémoslo

Esta es la propiedad central de BERT, y la que lo distingue de los embeddings estáticos
(word2vec, GloVe), donde cada palabra tiene un único vector fijo en una tabla.

Usemos la palabra inglesa `bank`, que tiene dos sentidos totalmente distintos:

- **S1:** *"I deposited money at the bank yesterday."* → banco financiero
- **S2:** *"We sat on the bank of the river and watched the sunset."* → orilla del río
- **S3:** *"The bank approved my loan application."* → banco financiero

Si los embeddings fueran estáticos, el vector de `bank` sería **idéntico** en las tres y las tres
similitudes valdrían 1. Vamos a medirlo.

In [ ]:
def tokens_y_vectores(texto):
    """Devuelve la lista de tokens y la matriz (n_tokens x 768) de sus embeddings."""
    codificado = tokenizador(texto, return_tensors="pt")
    with torch.no_grad():
        salida = bert(**codificado)
    tokens = tokenizador.convert_ids_to_tokens(codificado["input_ids"][0])
    return tokens, salida.last_hidden_state[0].numpy()


def similitud_coseno(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))


def vector_de_palabra(texto, palabra):
    tokens, vectores = tokens_y_vectores(texto)
    return vectores[tokens.index(palabra)]


S1 = "I deposited money at the bank yesterday."
S2 = "We sat on the bank of the river and watched the sunset."
S3 = "The bank approved my loan application."

v1 = vector_de_palabra(S1, "bank")
v2 = vector_de_palabra(S2, "bank")
v3 = vector_de_palabra(S3, "bank")

print("Similitud coseno entre los vectores de la MISMA palabra 'bank':\n")
print(f"  S1 vs S3  (banco financiero  vs  banco financiero) = {similitud_coseno(v1, v3):.3f}")
print(f"  S1 vs S2  (banco financiero  vs  orilla del río  ) = {similitud_coseno(v1, v2):.3f}")
print(f"  S3 vs S2  (banco financiero  vs  orilla del río  ) = {similitud_coseno(v3, v2):.3f}")

**Este es el resultado más importante del notebook.** La misma cadena de caracteres,
`bank`, recibe vectores claramente distintos según el sentido en que se use: alta similitud entre
los dos usos financieros, mucho más baja entre el financiero y el geográfico. BERT **desambigua
por contexto**, cosa que un embedding estático no puede hacer por construcción.

Piensa en lo que esto habilita para análisis de texto en ciencias sociales: `"banco"` (institución
financiera / banca de un parque), `"partido"` (organización política / encuentro deportivo /
dividido), `"plancha"` (electrodoméstico / plataforma política). Con embeddings estáticos, todos
esos sentidos se promedian en un solo vector y se pierden.

### 2.1 Y sin embargo, no hay "un vector de la oración"

BERT nos entregó **un vector por token**. Si queremos comparar dos oraciones completas necesitamos
un solo vector por oración. La forma más común de obtenerlo es el ***mean pooling***: promediar
los vectores de todos los tokens.

Hay un detalle que sí importa. Cuando procesamos varias oraciones juntas, las más cortas se
rellenan con tokens `[PAD]` para que todas tengan el mismo largo. Esos tokens no significan nada,
así que **no deben entrar en el promedio**. Para eso sirve la `attention_mask`: vale 1 en los
tokens reales y 0 en el relleno. Sin la máscara, una oración corta en un lote de oraciones largas
quedaría con su significado diluido en decenas de ceros.

In [ ]:
def embedding_oracion_bert(oraciones, tamano_lote=32):
    """Mean pooling de los embeddings de token, ignorando el relleno ([PAD])."""
    if isinstance(oraciones, str):
        oraciones = [oraciones]

    resultados = []
    for inicio in range(0, len(oraciones), tamano_lote):
        lote = oraciones[inicio:inicio + tamano_lote]
        codificado = tokenizador(lote, padding=True, truncation=True, return_tensors="pt")

        with torch.no_grad():
            salida = bert(**codificado)

        vectores_token = salida.last_hidden_state                       # (lote, tokens, 768)
        mascara = codificado["attention_mask"].unsqueeze(-1) \
                                              .expand(vectores_token.size()).float()

        suma = torch.sum(vectores_token * mascara, dim=1)               # suma solo tokens reales
        conteo = torch.clamp(mascara.sum(dim=1), min=1e-9)              # cuántos eran reales
        resultados.append((suma / conteo).numpy())

    return np.vstack(resultados)


prueba = embedding_oracion_bert(["The bank approved my loan.", "Short."])
print("Forma:", prueba.shape, " (2 oraciones x 768 dimensiones)")

## 3. Primera prueba cualitativa: matriz de similitud

Tomemos 11 mensajes agrupados en cuatro temas evidentes (celulares, clima, comida y salud, y
preguntar la edad). Si el método funciona, la matriz de similitud debe mostrar **bloques
brillantes en la diagonal**: cada mensaje se parece a los de su tema y no a los demás.

In [ ]:
mensajes = [
    # Celulares
    "I like my phone",
    "My phone is not good.",
    "Your cellphone looks great.",
    # Clima
    "Will it snow tomorrow?",
    "Recently a lot of hurricanes have hit the US",
    "Global warming is real",
    # Comida y salud
    "An apple a day, keeps the doctors away",
    "Eating strawberries is healthy",
    "Is paleo better than keto?",
    # Preguntar la edad
    "How old are you?",
    "what is your age?",
]

def matriz_similitud_coseno(vectores):
    normas = np.linalg.norm(vectores, axis=1, keepdims=True)
    normalizados = vectores / normas
    return np.round(np.inner(normalizados, normalizados), 4)


def graficar_similitud(etiquetas, vectores, titulo, ax=None, vmin=0):
    sim = matriz_similitud_coseno(vectores)
    if ax is None:
        plt.figure(figsize=(9, 7))
        ax = plt.gca()
    sns.heatmap(sim, xticklabels=etiquetas, yticklabels=etiquetas,
                vmin=vmin, vmax=1, cmap="YlOrRd", annot=False, ax=ax)
    ax.set_xticklabels(etiquetas, rotation=90)
    ax.set_title(titulo)
    return sim

In [ ]:
emb_bert = embedding_oracion_bert(mensajes)
sim_bert = graficar_similitud(mensajes, emb_bert,
                              "BERT + mean pooling (embeddings de token promediados)")
plt.show()

fuera_diagonal = sim_bert[~np.eye(len(mensajes), dtype=bool)]
print(f"Similitudes fuera de la diagonal: mínimo {fuera_diagonal.min():.3f}, "
      f"máximo {fuera_diagonal.max():.3f}, promedio {fuera_diagonal.mean():.3f}")

**Mira el rango, no solo el dibujo.** Los bloques temáticos se alcanzan a distinguir, pero el
mapa completo está tibio: **ningún par baja de ~0.47 de similitud**, aunque no tenga absolutamente
nada que ver ("Will it snow tomorrow?" vs. "Is paleo better than keto?"), y el promedio general
ronda 0.57. Compáralo con lo que veremos en la sección 5: el mismo cálculo con SBERT baja a un
promedio de ~0.13 y llega a valores negativos.

Este fenómeno tiene nombre: **anisotropía**. Los embeddings de BERT no ocupan todo el espacio de
768 dimensiones, sino un cono estrecho dentro de él, así que cualquier par de vectores apunta más
o menos hacia la misma dirección. La consecuencia práctica es grave: si los pares no relacionados
ya empiezan en 0.47, **no puedes fijar un umbral** confiable que separe "relacionados" de "no
relacionados".

Y hay una razón de fondo: **BERT nunca fue entrenado para esta tarea.** Se entrenó para predecir
palabras ocultas dentro de una oración, no para que el promedio de sus tokens sirviera como medida
de similitud entre oraciones. Estamos usando la herramienta para algo que no le pidieron.

## 4. Segunda prueba, esta vez cuantitativa: STS-Benchmark

Una matriz de colores no es evidencia suficiente. Necesitamos un **estándar externo**, y para eso
existe **STS-Benchmark** (*Semantic Textual Similarity*): miles de pares de oraciones a los que
**anotadores humanos** asignaron una calificación de similitud de **0 a 5**, según reglas
explícitas (ver la tabla 1 de [Cer et al., 2017](https://aclanthology.org/S17-2001.pdf)):

| Puntaje | Criterio del anotador |
|---|---|
| 5 | Son completamente equivalentes: significan lo mismo |
| 4 | Son mayormente equivalentes, difieren en detalles sin importancia |
| 3 | Son a grandes rasgos equivalentes, pero difieren en información importante |
| 2 | No son equivalentes, pero comparten tema |
| 1 | No son equivalentes, pero tratan del mismo asunto general |
| 0 | No tienen nada que ver |

Nuestro procedimiento de evaluación es el de siempre en aprendizaje supervisado, aunque aquí no
estemos entrenando nada: **comparar la predicción del modelo contra una etiqueta de referencia**.
La predicción es la similitud coseno; la etiqueta es el juicio humano; la métrica de acuerdo es la
correlación.

> **Sobre los datos.** Esta carpeta incluye `stsbenchmark_muestra.csv`, una copia de 200 pares
> del conjunto de prueba. El notebook la usa si está disponible, y si no —por ejemplo, si abriste
> este archivo suelto en Google Colab— descarga el conjunto completo con el paquete `datasets`.

> **Nota sobre la escala.** Dividimos el puntaje humano entre 5 para dejarlo en [0, 1] y que sea
> visualmente comparable con el coseno. Esto **no** afecta a la correlación de Pearson (es una
> transformación lineal), solo hace legible la gráfica. Y una advertencia conceptual: la escala
> humana y el coseno no son la misma cosa —un 2.5 humano no "debe" corresponder a un coseno de
> 0.5—, por eso lo que evaluamos es la **correlación**, es decir, si ambos ordenan igual los
> pares, no si coinciden número a número.

In [ ]:
RUTA_LOCAL = "stsbenchmark_muestra.csv"   # copia de 200 pares incluida en esta carpeta

# Si tienes la carpeta completa (repositorio clonado), usamos la copia local: es
# instantánea y no depende de la red. Si no está --por ejemplo, si abriste este
# notebook suelto en Google Colab-- bajamos el conjunto completo de Hugging Face.
if os.path.exists(RUTA_LOCAL):
    sts = pd.read_csv(RUTA_LOCAL)
    print(f"Usando la copia local: {len(sts)} pares.")
    print("(Para trabajar con los 1,379 pares del conjunto de prueba completo, "
          "renombra o borra el CSV y vuelve a correr esta celda.)")
else:
    from datasets import load_dataset
    datos_sts = load_dataset("mteb/stsbenchmark-sts")["test"]
    sts = pd.DataFrame({
        "sentence1": datos_sts["sentence1"],
        "sentence2": datos_sts["sentence2"],
        "score":     datos_sts["score"],
    })
    print(f"Descargado de Hugging Face: {len(sts)} pares.")

N_EJEMPLOS = 200
sts = sts.head(N_EJEMPLOS).copy()
sts["score"] = sts["score"] / 5.0          # llevamos el juicio humano a [0, 1]

print(f"\nTrabajaremos con {len(sts)} pares.")
sts.head(8)

In [ ]:
def coseno_por_fila(A, B):
    """Similitud coseno entre A[i] y B[i], vectorizada."""
    return np.sum(A * B, axis=1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1))


A_bert = embedding_oracion_bert(list(sts["sentence1"]))
B_bert = embedding_oracion_bert(list(sts["sentence2"]))
sts["coseno_bert"] = coseno_por_fila(A_bert, B_bert)

sts[["sentence1", "sentence2", "score", "coseno_bert"]].head(8)

In [ ]:
r_bert, p_bert = scipy.stats.pearsonr(sts["score"], sts["coseno_bert"])
rho_bert, _ = scipy.stats.spearmanr(sts["score"], sts["coseno_bert"])

print("BERT + mean pooling")
print(f"  Correlación de Pearson  = {r_bert:.3f}   (p = {p_bert:.2e})")
print(f"  Correlación de Spearman = {rho_bert:.3f}")
print(f"  Rango de los cosenos    = [{sts['coseno_bert'].min():.3f}, {sts['coseno_bert'].max():.3f}]"
      f"   promedio = {sts['coseno_bert'].mean():.3f}")

Guarda ese número. Volveremos a él en un minuto.

> **Un apunte metodológico.** Reportamos Pearson y Spearman juntos a propósito. Pearson mide
> asociación **lineal**; Spearman mide si ambos **ordenan igual** los pares, sin exigir linealidad.
> Para juzgar un sistema de recuperación de información, lo que de verdad importa es el orden
> —qué documento sale primero—, así que Spearman suele ser la métrica más honesta. Es la misma
> distinción que discutimos al evaluar modelos en las sesiones de regresión.

## 5. Un modelo entrenado para la tarea: SBERT / dual encoder

La alternativa es un modelo **entrenado explícitamente para producir embeddings de oración
comparables**. La familia SBERT (*Sentence-BERT*) parte de un BERT y lo reentrena con una
arquitectura de **dual encoder**: se le muestran pares de oraciones y se ajustan los pesos para
que los pares similares queden cerca y los disímiles lejos (aprendizaje contrastivo). El resultado
es un modelo cuyo espacio vectorial **sí** está construido para que la distancia signifique algo.

Usamos `all-MiniLM-L6-v2`, el mismo del notebook 04: 6 capas en vez de 12, 384 dimensiones en vez
de 768, 87 MB en vez de 420. Es decir, es un modelo **más chico y más rápido** que BERT. Veamos si
eso le impide ganar.

In [ ]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Dimensión del embedding: {sbert.get_sentence_embedding_dimension()}")

In [ ]:
emb_sbert = sbert.encode(mensajes)

fig, axes = plt.subplots(1, 2, figsize=(19, 7))
graficar_similitud(mensajes, emb_bert, "BERT + mean pooling", ax=axes[0])
sim_sbert = graficar_similitud(mensajes, emb_sbert, "SBERT (dual encoder)", ax=axes[1])
plt.tight_layout()
plt.show()

for nombre, sim in [("BERT ", sim_bert), ("SBERT", sim_sbert)]:
    fuera = sim[~np.eye(len(mensajes), dtype=bool)]
    print(f"{nombre}: similitudes fuera de la diagonal en "
          f"[{fuera.min():.3f}, {fuera.max():.3f}], promedio {fuera.mean():.3f}")

**Compara los dos mapas lado a lado.** En el de SBERT los cuatro bloques temáticos aparecen
nítidos sobre un fondo frío, y el rango de similitudes se abre muchísimo. Eso es lo que hace
utilizable un umbral del tipo *"considera relacionados los pares con similitud > 0.5"*: con BERT
crudo ese umbral no separaría nada.

In [ ]:
A_sbert = sbert.encode(list(sts["sentence1"]))
B_sbert = sbert.encode(list(sts["sentence2"]))
sts["coseno_sbert"] = coseno_por_fila(A_sbert, B_sbert)

r_sbert, p_sbert = scipy.stats.pearsonr(sts["score"], sts["coseno_sbert"])
rho_sbert, _ = scipy.stats.spearmanr(sts["score"], sts["coseno_sbert"])

resumen = pd.DataFrame({
    "Pearson":              [r_bert, r_sbert],
    "Spearman":             [rho_bert, rho_sbert],
    "coseno mínimo":        [sts["coseno_bert"].min(), sts["coseno_sbert"].min()],
    "coseno máximo":        [sts["coseno_bert"].max(), sts["coseno_sbert"].max()],
    "coseno promedio":      [sts["coseno_bert"].mean(), sts["coseno_sbert"].mean()],
    "dimensiones":          [bert.config.hidden_size, sbert.get_sentence_embedding_dimension()],
}, index=["BERT + mean pooling", "SBERT (all-MiniLM-L6-v2)"]).round(3)

resumen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)

for ax, columna, titulo, r in [
    (axes[0], "coseno_bert",  "BERT + mean pooling", r_bert),
    (axes[1], "coseno_sbert", "SBERT (dual encoder)", r_sbert),
]:
    ax.scatter(sts["score"], sts[columna], alpha=0.6, s=28)
    ax.set_xlabel("Juicio humano de similitud (0 = nada que ver, 1 = equivalentes)")
    ax.set_title(f"{titulo}\nPearson = {r:.3f}")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.25, 1.05)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Similitud coseno del modelo")
plt.suptitle("Acuerdo entre el modelo y el estándar humano (STS-Benchmark)")
plt.tight_layout()
plt.show()

**Las dos nubes de puntos cuentan toda la historia.**

- La de **BERT** es una banda horizontal aplastada contra la parte superior: sin importar si los
  humanos dijeron 0 o 5, el modelo responde siempre "se parecen bastante". La correlación es baja
  no porque el modelo se equivoque de dirección, sino porque **casi no varía**.
- La de **SBERT** se estira en diagonal: cuando los humanos dicen 0 el modelo da valores bajos
  —incluso negativos— y cuando dicen 5 da valores altos.

Y el punto que conviene subrayar en clase: **el modelo que gana es el más chico de los dos.**
SBERT tiene la mitad de capas y la mitad de dimensiones que BERT, y aun así lo supera por un
margen enorme en esta tarea. Lo determinante no fue el tamaño, sino que **el objetivo de
entrenamiento coincidiera con la tarea de uso**. Es la misma lección que en cualquier otro modelo
del módulo: un modelo bien especificado para la pregunta le gana a uno más grande mal aplicado.

## 6. ¿Y si mejoramos el uso de BERT en lugar de cambiar de modelo?

Antes de dar por perdido a BERT, probemos una alternativa común: en vez de promediar todos los
tokens, usar el vector del token especial `[CLS]`, que en teoría resume la oración completa
(es el que se usa como entrada a la capa de clasificación cuando se hace *fine-tuning*).

In [ ]:
def embedding_cls(oraciones, tamano_lote=32):
    """Usa el vector del token [CLS] (posición 0) como representación de la oración."""
    resultados = []
    for inicio in range(0, len(oraciones), tamano_lote):
        codificado = tokenizador(oraciones[inicio:inicio + tamano_lote],
                                 padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad():
            salida = bert(**codificado)
        resultados.append(salida.last_hidden_state[:, 0, :].numpy())
    return np.vstack(resultados)


A_cls = embedding_cls(list(sts["sentence1"]))
B_cls = embedding_cls(list(sts["sentence2"]))
sts["coseno_cls"] = coseno_por_fila(A_cls, B_cls)

r_cls, _ = scipy.stats.pearsonr(sts["score"], sts["coseno_cls"])
rho_cls, _ = scipy.stats.spearmanr(sts["score"], sts["coseno_cls"])

print(f"BERT [CLS]          : Pearson = {r_cls:.3f}   Spearman = {rho_cls:.3f}")
print(f"BERT + mean pooling : Pearson = {r_bert:.3f}   Spearman = {rho_bert:.3f}")
print(f"SBERT               : Pearson = {r_sbert:.3f}   Spearman = {rho_sbert:.3f}")

Compara los tres renglones antes de leer la conclusión.

El resultado es todavía más contundente de lo esperado: **`[CLS]` no solo no mejora, sino que
resulta *peor* que el promedio de tokens** —su correlación queda prácticamente en cero, es decir,
sus similitudes no tienen relación alguna con el juicio humano.

Y esa es la razón por la que existe SBERT: **el problema no era cómo agregábamos los tokens**. Ni
promediarlos ni tomar `[CLS]` produce un espacio donde la distancia signifique similitud
semántica, porque el modelo nunca fue entrenado para eso. `[CLS]` resume la oración *para la tarea
con la que se entrenó* (predecir si dos fragmentos van seguidos, y palabras enmascaradas), que no
es medir similitud. Cambiar el método de agregación no arregla un objetivo de entrenamiento
equivocado; hay que reentrenar el modelo con pares de oraciones, que es exactamente lo que hace
SBERT.

## Para pensar

1. En la sección 2 mostramos que `bank` recibe vectores distintos según el contexto. Diseña, en
   dos o tres líneas, un experimento equivalente **en español** con una palabra ambigua
   (`banco`, `partido`, `capital`, `sierra`). ¿Con qué modelo tendrías que hacerlo, y por qué no
   sirve `bert-base-uncased`? (Revisa la salida de la sección 1 antes de responder.)

2. La `attention_mask` del *mean pooling* excluye los tokens `[PAD]` del promedio. Describe qué
   pasaría con el embedding de la oración `"Short."` si la procesáramos en un lote junto a
   oraciones de 200 tokens **sin** usar la máscara. ¿Se parecería más o menos a las demás
   oraciones del lote? ¿Por qué eso es un problema para la matriz de similitud?

3. En la sección 4, la correlación de Pearson y la de Spearman de BERT no coinciden. Explica qué
   significa esa diferencia y cuál de las dos usarías para decidir si un sistema de búsqueda
   semántica sirve para tu proyecto.

4. Los embeddings de BERT ocupan un cono estrecho del espacio (anisotropía) y por eso todas las
   similitudes salen altas. Un compañero propone "arreglarlo" reescalando las similitudes a
   [0, 1] con un min-max sobre el conjunto de prueba. ¿Resolvería el problema de fondo? ¿Qué
   pasaría al llegar un par de oraciones nuevo, fuera de ese conjunto?

5. **Ejercicio.** Aumenta `N_EJEMPLOS` de 200 a 1,000 (necesitas el paquete `datasets`, la copia
   local solo trae 200) y vuelve a correr la comparación. ¿Cambian mucho las correlaciones?
   ¿Qué te dice eso sobre la confiabilidad de las conclusiones que sacamos con 200 pares?

6. **Ejercicio.** SBERT ganó con 384 dimensiones contra las 768 de BERT. Prueba
   `all-mpnet-base-v2` (768 dimensiones, ~420 MB, el modelo de mayor calidad de
   `sentence-transformers`) y mide cuánto gana en correlación frente a `all-MiniLM-L6-v2`.
   Con el tiempo de cómputo que también vas a medir, ¿lo justificarías en un proyecto que tiene
   que procesar un millón de documentos?

---

**Siguiente bloque:** `06_Using_Embeddings_in_RAG_Inference.ipynb`, donde estos embeddings de
oración se convierten en un sistema de recuperación de información, el primer paso de RAG.